# Goal Scorer Prediction — LightGBM + XGBoost + CatBoost Ensemble

In [ ]:
import warnings
import time

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from scipy.stats import rankdata

warnings.filterwarnings("ignore")

t0 = time.time()
N_FOLDS = 5
RANDOM_STATE = 42
DATA_DIR = "/home/neeraj/Desktop/Coding/Datathon"

## 1. Load Data

In [ ]:
print("Loading datasets...")

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")

train["player_id"] = train["appearance_id"].str.split("_").str[1].astype(int)
train["game_id"]   = train["appearance_id"].str.split("_").str[0].astype(int)
test["player_id"]  = test["appearance_id"].str.split("_").str[1].astype(int)
test["game_id"]    = test["appearance_id"].str.split("_").str[0].astype(int)

y = train["scored_flag"].astype(int)

print(f"  Train shape: {train.shape} | Test shape: {test.shape}")

## 2. Feature Engineering

In [ ]:
print("Generating features...")


def b2i(s):
    return s.map({True: 1, False: 0, "True": 1, "False": 0}).fillna(0).astype(int)


for df in [train, test]:
    df["total_goals"]    = df["home_club_goals"] + df["away_club_goals"]
    df["is_home"]        = (df["home_away"] == "HOME").astype(int)
    df["own_team_goals"] = np.where(df["is_home"] == 1, df["home_club_goals"], df["away_club_goals"])
    df["opp_goals"]      = np.where(df["is_home"] == 1, df["away_club_goals"], df["home_club_goals"])
    df["player_club_id"] = np.where(df["is_home"] == 1, df["home_club_id"],    df["away_club_id"])
    df["opp_club_id"]    = np.where(df["is_home"] == 1, df["away_club_id"],    df["home_club_id"])
    df["month"]          = pd.to_datetime(df["date"], errors="coerce").dt.month
    df["dow"]            = pd.to_datetime(df["date"], errors="coerce").dt.dayofweek

    xg  = df["avg_xG"].fillna(0)
    npx = df["avg_npxG"].fillna(0)
    sh  = df["avg_shots"].fillna(0)
    xa  = df["avg_xA"].fillna(0)
    xc  = df["avg_xGChain"].fillna(0)
    mr  = df["minutes_ratio"]

    df["xG_x_min"]    = xg * mr
    df["npxG_x_min"]  = npx * mr
    df["shots_x_min"] = sh * mr
    df["xGC_x_min"]   = xc * mr
    df["xA_x_min"]    = xa * mr
    df["xG2"]         = xg ** 2
    df["npxG2"]       = npx ** 2
    df["off_idx"]     = xg + npx + sh * 0.05
    df["fin_vs_cre"]  = xg - xa
    df["xG_share"]    = xg / (xc + 0.01)
    df["shot_qual"]   = xg / (sh + 0.01)
    df["mv_x_xG"]     = df["log_market_value"].fillna(0) * xg
    df["xG_att"]      = xg * b2i(df["is_attacker"])
    df["xG_mid"]      = xg * b2i(df["is_midfielder"])
    df["xG_def"]      = xg * b2i(df["is_defender"])
    df["home_att"]    = df["is_home"] * b2i(df["is_attacker"])
    df["xG_per_tg"]   = xg / (df["own_team_goals"] + 1)
    df["intl_gpc"]    = df["international_goals"].fillna(0) / (df["international_caps"].fillna(0) + 1)
    df["log_att"]     = np.log1p(df["attendance"].fillna(0))
    df["xG_miss"]     = df["avg_xG"].isna().astype(int)
    df["has_us"]      = b2i(df["has_understat"])
    df["age2"]        = df["age"] ** 2

    bool_cols = [
        "full_match_flag", "starter_flag", "substitute_flag", "card_flag",
        "prime_age_flag", "veteran_flag", "has_national_team_experience",
        "finisher_flag", "creative_player_flag",
        "is_goalkeeper", "is_defender", "is_midfielder", "is_attacker",
    ]
    for c in bool_cols:
        if c in df.columns:
            df[c] = b2i(df[c])

print("  Feature engineering complete.")

## 3. Target Encoding

In [ ]:
print("Creating target encodings...")

kf    = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
folds = list(kf.split(train, y))
gm    = y.mean()


def target_encode(tr, te_df, col, target, folds, sm=30):
    enc_tr = pd.Series(np.nan, index=tr.index, dtype=float)
    for _, (ti, vi) in enumerate(folds):
        s = target.iloc[ti].groupby(tr[col].iloc[ti]).agg(["mean", "count"])
        g = target.iloc[ti].mean()
        s["s"] = (s["count"] * s["mean"] + sm * g) / (s["count"] + sm)
        enc_tr.iloc[vi] = tr[col].iloc[vi].map(s["s"])
    enc_tr = enc_tr.fillna(gm)

    fs = target.groupby(tr[col]).agg(["mean", "count"])
    fs["s"] = (fs["count"] * fs["mean"] + sm * gm) / (fs["count"] + sm)
    enc_te = te_df[col].map(fs["s"]).fillna(gm)
    return enc_tr, enc_te


smoothed_cols = [
    ("player_id", 30), ("player_club_id", 50), ("opp_club_id", 50),
    ("sub_position", 50), ("game_id", 20),
]
for col, sm in smoothed_cols:
    if col in train.columns:
        train[f"{col}_te"], test[f"{col}_te"] = target_encode(train, test, col, y, folds, sm)

categorical_cols = [
    "country_name", "competition_type", "confederation", "foot", "position",
    "country_of_citizenship", "stadium", "referee",
]
for col in categorical_cols:
    if col in train.columns:
        train[f"{col}_te"], test[f"{col}_te"] = target_encode(train, test, col, y, folds, 50)

for col in ["player_id", "player_club_id", "game_id"]:
    cn = train[col].value_counts()
    train[f"{col}_cnt"] = train[col].map(cn).astype(float)
    test[f"{col}_cnt"]  = test[col].map(cn).fillna(1).astype(float)

for df in [train, test]:
    df["pte_x_min"]  = df["player_id_te"] * df["minutes_ratio"]
    df["pte_x_xG"]   = df["player_id_te"] * df["avg_xG"].fillna(0)
    df["cte_x_ote"]  = df["player_club_id_te"] * df["opp_club_id_te"]
    df["spte_x_min"] = df["sub_position_te"] * df["minutes_ratio"]

print("  Target encoding complete.")

## 4. Feature Selection & Matrix Preparation

In [ ]:
feats = [
    "season", "minutes_played", "yellow_cards", "red_cards", "disciplinary_points",
    "home_club_goals", "away_club_goals", "attendance", "market_value_before_match",
    "age", "height_in_cm", "international_caps", "international_goals",
    "highest_market_value_in_eur", "market_value_ratio", "avg_xG", "avg_xA",
    "avg_shots", "avg_key_passes", "avg_xGChain", "avg_xGBuildup", "avg_npxG",
    "log_market_value", "value_pct_of_peak", "minutes_ratio", "goal_diff_abs",
    "goal_per_cap", "xG_to_xA_ratio", "full_match_flag", "starter_flag",
    "substitute_flag", "card_flag", "prime_age_flag", "veteran_flag",
    "has_national_team_experience", "finisher_flag", "creative_player_flag",
    "is_goalkeeper", "is_defender", "is_midfielder", "is_attacker",
    "month", "dow", "is_home", "total_goals", "own_team_goals", "opp_goals",
    "xG_x_min", "npxG_x_min", "shots_x_min", "xGC_x_min", "xA_x_min", "xG2", "npxG2",
    "off_idx", "fin_vs_cre", "xG_share", "shot_qual", "mv_x_xG", "xG_att", "xG_mid",
    "xG_def", "home_att", "xG_per_tg", "intl_gpc", "log_att", "xG_miss", "has_us", "age2",
    "player_id_te", "player_club_id_te", "opp_club_id_te", "sub_position_te", "game_id_te",
    "country_name_te", "competition_type_te", "confederation_te", "foot_te",
    "position_te", "country_of_citizenship_te", "stadium_te", "referee_te",
    "player_id_cnt", "player_club_id_cnt", "game_id_cnt",
    "pte_x_min", "pte_x_xG", "cte_x_ote", "spte_x_min",
]
feats = [f for f in feats if f in train.columns]

X      = np.nan_to_num(train[feats].values.astype(np.float32), nan=0, posinf=0, neginf=0)
X_test = np.nan_to_num(test[feats].values.astype(np.float32),  nan=0, posinf=0, neginf=0)
yv     = y.values

print(f"  Feature matrix: {X.shape[1]} features | {X.shape[0]} train rows | {X_test.shape[0]} test rows")

## 5. Model Training
### 5a. LightGBM

In [ ]:
print("Training LightGBM...")

lgb_params = {
    "objective": "binary",
    "metric": "average_precision",
    "verbosity": -1,
    "n_jobs": -1,
    "seed": RANDOM_STATE,
    "learning_rate": 0.05,
    "num_leaves": 255,
    "max_depth": -1,
    "min_child_samples": 50,
    "subsample": 0.85,
    "subsample_freq": 1,
    "colsample_bytree": 0.7,
    "reg_alpha": 2.0,
    "reg_lambda": 2.0,
    "scale_pos_weight": 10.7,
    "max_bin": 255,
}

oof_lgb  = np.zeros(len(X))
test_lgb = np.zeros(len(X_test))

for i, (ti, vi) in enumerate(folds):
    dt = lgb.Dataset(X[ti], label=yv[ti], feature_name=feats)
    dv = lgb.Dataset(X[vi], label=yv[vi], feature_name=feats, reference=dt)
    m = lgb.train(
        lgb_params, dt, 2000, valid_sets=[dv],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
    )
    oof_lgb[vi] = m.predict(X[vi])
    test_lgb   += m.predict(X_test) / N_FOLDS
    fold_ap     = average_precision_score(yv[vi], oof_lgb[vi])
    print(f"  Fold {i + 1}/{N_FOLDS} | AP = {fold_ap:.4f}")

lgb_ap = average_precision_score(yv, oof_lgb)
print(f"LightGBM OOF AP: {lgb_ap:.4f}")

pd.DataFrame({"appearance_id": test["appearance_id"], "scored_flag": test_lgb}).to_csv(
    f"{DATA_DIR}/solution_lgb.csv", index=False
)
print("Saved: solution_lgb.csv")

### 5b. XGBoost

In [ ]:
print("Training XGBoost...")

xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "verbosity": 0,
    "nthread": -1,
    "seed": RANDOM_STATE,
    "learning_rate": 0.05,
    "max_depth": 8,
    "min_child_weight": 50,
    "subsample": 0.85,
    "colsample_bytree": 0.7,
    "reg_alpha": 2.0,
    "reg_lambda": 2.0,
    "scale_pos_weight": 10.7,
    "tree_method": "hist",
}

oof_xgb  = np.zeros(len(X))
test_xgb = np.zeros(len(X_test))

for i, (ti, vi) in enumerate(folds):
    dx = xgb.DMatrix(X[ti], label=yv[ti], feature_names=feats)
    dv = xgb.DMatrix(X[vi], label=yv[vi], feature_names=feats)
    m = xgb.train(
        xgb_params, dx, 2000,
        evals=[(dv, "val")],
        early_stopping_rounds=100,
        verbose_eval=False,
    )
    oof_xgb[vi] = m.predict(dv)
    test_xgb   += m.predict(xgb.DMatrix(X_test, feature_names=feats)) / N_FOLDS
    fold_ap     = average_precision_score(yv[vi], oof_xgb[vi])
    print(f"  Fold {i + 1}/{N_FOLDS} | AP = {fold_ap:.4f}")

xgb_ap = average_precision_score(yv, oof_xgb)
print(f"XGBoost OOF AP: {xgb_ap:.4f}")

pd.DataFrame({"appearance_id": test["appearance_id"], "scored_flag": test_xgb}).to_csv(
    f"{DATA_DIR}/solution_xgb.csv", index=False
)
print("Saved: solution_xgb.csv")

### 5c. CatBoost

In [ ]:
print("Training CatBoost...")

oof_cb  = np.zeros(len(X))
test_cb = np.zeros(len(X_test))

for i, (ti, vi) in enumerate(folds):
    m = cb.CatBoostClassifier(
        iterations=2000,
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=3.0,
        random_seed=RANDOM_STATE,
        verbose=False,
        early_stopping_rounds=100,
        eval_metric="Logloss",
        auto_class_weights="Balanced",
        task_type="CPU",
        thread_count=-1,
    )
    m.fit(X[ti], yv[ti], eval_set=(X[vi], yv[vi]))
    oof_cb[vi] = m.predict_proba(X[vi])[:, 1]
    test_cb   += m.predict_proba(X_test)[:, 1] / N_FOLDS
    fold_ap    = average_precision_score(yv[vi], oof_cb[vi])
    print(f"  Fold {i + 1}/{N_FOLDS} | AP = {fold_ap:.4f}")

cb_ap = average_precision_score(yv, oof_cb)
print(f"CatBoost OOF AP: {cb_ap:.4f}")

pd.DataFrame({"appearance_id": test["appearance_id"], "scored_flag": test_cb}).to_csv(
    f"{DATA_DIR}/solution_cb.csv", index=False
)
print("Saved: solution_cb.csv")

## 6. Ensemble Optimization

In [ ]:
print("Optimizing ensemble weights...")

# --- Probability blending ---
best_ap, best_w = 0, None
for w1 in np.arange(0.2, 0.8, 0.05):
    for w2 in np.arange(0.1, 0.6, 0.05):
        w3 = 1 - w1 - w2
        if w3 < 0.05:
            continue
        blend = w1 * oof_lgb + w2 * oof_xgb + w3 * oof_cb
        ap = average_precision_score(yv, blend)
        if ap > best_ap:
            best_ap, best_w = ap, (w1, w2, w3)

print(
    f"  Prob blend  — LGB={best_w[0]:.2f}  XGB={best_w[1]:.2f}  CB={best_w[2]:.2f}"
    f"  ->  AP={best_ap:.4f}"
)

# --- Rank blending ---
lr = rankdata(oof_lgb) / len(yv)
xr = rankdata(oof_xgb) / len(yv)
cr = rankdata(oof_cb)  / len(yv)

best_rap, best_rw = 0, None
for w1 in np.arange(0.2, 0.8, 0.05):
    for w2 in np.arange(0.1, 0.6, 0.05):
        w3 = 1 - w1 - w2
        if w3 < 0.05:
            continue
        blend = w1 * lr + w2 * xr + w3 * cr
        ap = average_precision_score(yv, blend)
        if ap > best_rap:
            best_rap, best_rw = ap, (w1, w2, w3)

print(
    f"  Rank blend  — LGB={best_rw[0]:.2f}  XGB={best_rw[1]:.2f}  CB={best_rw[2]:.2f}"
    f"  ->  AP={best_rap:.4f}"
)

# --- Select best strategy ---
if best_rap > best_ap:
    print("  Selected: rank blending")
    final = (
        best_rw[0] * (rankdata(test_lgb) / len(test_lgb))
        + best_rw[1] * (rankdata(test_xgb) / len(test_xgb))
        + best_rw[2] * (rankdata(test_cb)  / len(test_cb))
    )
    f_ap = best_rap
else:
    print("  Selected: probability blending")
    final = best_w[0] * test_lgb + best_w[1] * test_xgb + best_w[2] * test_cb
    f_ap  = best_ap

sub = pd.DataFrame({"appearance_id": test["appearance_id"], "scored_flag": final})
sub.to_csv(f"{DATA_DIR}/ensemble.csv", index=False)
print(f"Saved: ensemble.csv  |  Ensemble AP = {f_ap:.4f}")

## 7. Summary

In [ ]:
elapsed = (time.time() - t0) / 60

print("=" * 50)
print("  Run Summary")
print("=" * 50)
print(f"  LightGBM OOF AP : {lgb_ap:.4f}")
print(f"  XGBoost  OOF AP : {xgb_ap:.4f}")
print(f"  CatBoost OOF AP : {cb_ap:.4f}")
print(f"  Ensemble AP     : {f_ap:.4f}")
print(f"  Total Runtime   : {elapsed:.1f} minutes")
print("=" * 50)